[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benjamalegni/ML-luka/blob/main/02-deep-learning/01-tpe-eurosat-satelital/TPE_Redes_Neuronales.ipynb)

**Datos para reproducir el trabajo:** usar la selección EuroSAT RGB de la cátedra (10 clases, 12.000 imágenes). Colocar la carpeta `EuroSAT_TP` o el archivo `EuroSAT_TP.zip` en la raíz de Mi unidad de Google Drive. También se puede subir el ZIP a `/content/EuroSAT_TP.zip` desde el panel de archivos de Colab. La ejecución guardada corresponde a la selección de 12.000 imágenes; el dataset EuroSAT completo tiene 27.000.

# 1. Exploración y preparación de los datos

In [3]:
from pathlib import Path

# El ZIP subido a /content permite ejecutar sin montar Google Drive.
if not Path('/content/EuroSAT_TP.zip').is_file():
    from google.colab import drive
    drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
from pathlib import Path

# Cambiar estas rutas si el archivo de la cátedra está en otra ubicación.
drive_path = Path('/content/drive/MyDrive/EuroSAT_TP')
zip_candidates = [
    Path('/content/drive/MyDrive/EuroSAT_TP.zip'),
    Path('/content/EuroSAT_TP.zip'),
]
print('Carpeta disponible:', drive_path.is_dir())
print('ZIP disponible:', next((str(p) for p in zip_candidates if p.is_file()), 'ninguno'))

Carpeta disponible: True
ZIP disponible: ninguno


In [ ]:
import shutil
import zipfile

destino_local = Path('/content/EuroSAT_TP_local')
class_names_expected = {
    'AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial',
    'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake'
}

if drive_path.is_dir():
    if destino_local.exists():
        shutil.rmtree(destino_local)
    shutil.copytree(drive_path, destino_local)
else:
    zip_path = next((p for p in zip_candidates if p.is_file()), None)
    if zip_path is None:
        raise FileNotFoundError(
            'Se necesita EuroSAT_TP/ o EuroSAT_TP.zip en Mi unidad, '
            'o EuroSAT_TP.zip en /content. Consulte la instrucción inicial.'
        )
    if destino_local.exists():
        shutil.rmtree(destino_local)
    destino_local.mkdir()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destino_local / member.filename).resolve()
            if not target.is_relative_to(destino_local.resolve()):
                raise ValueError(f'Ruta insegura dentro del ZIP: {member.filename}')
        archive.extractall(destino_local)

# El ZIP puede contener una carpeta contenedora antes de las 10 clases.
if not class_names_expected.issubset({p.name for p in destino_local.iterdir() if p.is_dir()}):
    candidates = [p for p in destino_local.rglob('*') if p.is_dir()
                  and class_names_expected.issubset({q.name for q in p.iterdir() if q.is_dir()})]
    if len(candidates) != 1:
        raise ValueError('No se encontró una única carpeta con las 10 clases EuroSAT RGB.')
    destino_local = candidates[0]

print('Dataset:', destino_local)
print('Clases:', sorted(p.name for p in destino_local.iterdir() if p.is_dir()))

Este dataset incluye imagenes a color de distintos tipos de terrenos.

Cada clase representa un tipo distinto de cobertura del suelo, lo que hace que el dataset sea ideal para tareas de clasificación de imágenes en teledetección.

### Conteo de imágenes por clase y visualización de la distribución

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from collections import Counter

class_counts = Counter()
for folder in os.listdir(destino_local):
    folder_path = destino_local / folder
    if folder_path.is_dir():
        num_images = len([name for name in os.listdir(folder_path) if name.lower().endswith(('.', '.jpg', '.jpeg'))])
        class_counts[folder] = num_images

df_class_counts = pd.DataFrame(class_counts.items(), columns=['Class', 'Count'])
df_class_counts = df_class_counts.sort_values(by='Count', ascending=False)

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")
sns.barplot(x='Class', y='Count', data=df_class_counts, palette='viridis', hue='Class')
plt.title('Distribución de Imágenes por Clase en EuroSAT_TP')
plt.xlabel('Todas las clases tienen la misma cantidad de imagenes')
plt.ylabel('Número de Imágenes')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Visualización de imágenes de ejemplo por clase

In [ ]:
import random
from PIL import Image

classes = [f.name for f in destino_local.iterdir() if f.is_dir()]
classes.sort()

# Crear una figura para mostrar una imagen de cada clase
fig, axes = plt.subplots(1, len(classes), figsize=(20, 3))

for i, class_name in enumerate(classes):
    class_path = destino_local / class_name

    # Obtener todas las imágenes en la clase
    images_in_class = [f for f in class_path.iterdir() if f.suffix.lower() in ('.png', '.jpg', '.jpeg')]

    if images_in_class:
        # Seleccionar una imagen aleatoria
        random_image_path = random.choice(images_in_class)
        img = Image.open(random_image_path)

        axes[i].imshow(img)
        axes[i].set_title(class_name)
        axes[i].axis('off')
    else:
        axes[i].set_title(f'{class_name}\n(No images)')
        axes[i].axis('off')

plt.suptitle('Imágenes de Ejemplo por Clase del Dataset EuroSAT', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf

# constantes para splitting
SPLIT_RATIO = 0.3
SEED = 42
IMG_SIZE = (64, 64)
BATCH_SIZE = 32

# 70% Train y 30% Temp (Val + Test)
raw_train_ds, temp_ds = tf.keras.utils.image_dataset_from_directory(
    destino_local,
    validation_split=SPLIT_RATIO,
    subset="both",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# guardo class_names ANTES de transformar los datasets
class_names = raw_train_ds.class_names
num_classes = len(class_names)
print(f"Clases detectadas ({num_classes}): {class_names}")

# divido el 30% de test en 15% Val y 15% Test
temp_ds = temp_ds.cache()
val_size = int(len(temp_ds) * 0.5)
val_ds_raw = temp_ds.take(val_size)
test_ds_raw = temp_ds.skip(val_size)

# optimizaciones para GPU
AUTOTUNE = tf.data.AUTOTUNE
train_ds = raw_train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds_raw.prefetch(buffer_size=AUTOTUNE)
test_ds  = test_ds_raw.prefetch(buffer_size=AUTOTUNE)


In [ ]:
# como en las imagenes satelitales el norte es arbitrario, puedo hacer rotaciones  en 90°, 180°, 270°, espejados horizontales y verticales porque conservan 100% el significado semántico

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.1),
])

In [ ]:
# Get class names from the training dataset
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")
plt.suptitle('Sample Images from resized Dataset', y=1.02)
plt.tight_layout()
plt.show()

# 2. Desarrollo de la red neuronal propia

Estare utilizando redes neuronales convolucionales (CNN) debido a que esta arquitectura aprovecha patrones locales y comparte pesos, esto la hace ideal para trabajar con imagenes.


Utilizaré Keras, que abstrae los detalles de implementación y permite escribir el modelo de forma independiente del backend (TensorFlow, JAX o PyTorch).

## Arquitectura propuesta 1

In [ ]:
def build_baseline(n_classes=10):
    return tf.keras.Sequential([
        tf.keras.layers.Input((64, 64, 3)),
        tf.keras.layers.Rescaling(1./255),


        # input de 64x64 en RGB
        # la resolucion espacial disminuye en cada bloque, y aumenta la profundidad
        # en todos los bloques (menos en la cabeza) aplico un batch normalization antes de la relu - para estandarizar los datos de un bloque a otro
        # los 3 bloques siguen este flujo:
        # - Conv2d -> batchNormalization -> relu -> maxPooling2D -> spatialDropout

        # bloque 1 — textura y bordes (32x32)
        # busco capturar solo caracteristicas simples
        tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2), # divide la resolucion en 2
        tf.keras.layers.SpatialDropout2D(0.20),

        # bloque 2 — patrones de textura (16x16)
        tf.keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2), # divide la resolucion en 2
        tf.keras.layers.SpatialDropout2D(0.25),

        # bloque 3 — semántica de la clase (8x8)
        tf.keras.layers.Conv2D(128, 3, padding="same", use_bias=False),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Activation("relu"),
        tf.keras.layers.MaxPooling2D(2), # divide la resolucion en 2
        tf.keras.layers.SpatialDropout2D(0.30),

        # cabeza
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.40),
        tf.keras.layers.Dense(n_classes, activation="softmax"),
    ])

La arquitectura propuesta consta de 3 capas + la cabeza.

El flujo es Conv2d -> batchNormalization -> relu -> maxPooling2D -> spatialDropout.
Los valores en spatialDropout2D van aumentando para evitar el overfitting.

## Cabeza
Las transformaciones realizadas en la cabeza son:
- globalAveragePooling2D, donde calcula un promedio de sus valores de entrada
- luego se pasa por una layer densa (el evaluador),
- un dropout que apaga el 40% de las neuronas de la red neuronal densa
- una ultima capa de salida dense_1 que tiene el mismo numero de neuronas que las clases del EuroSat.

In [ ]:
build_baseline(n_classes=num_classes).summary()

In [ ]:
cnn = build_baseline(n_classes=10)

cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# CNN training
# history_cnn es el historial del entrenamiento, no la red en si
# probe anteriormente con 5 epocas, lo que resulto en metricas

epochs = 10
history_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs,
    verbose=1
)

In [ ]:
test_loss, test_acc = cnn.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.2f}")
print(f"Test loss: {test_loss:.2f}")

In [ ]:
# Graficar la pérdida y precisión del entrenamiento y validación
acc = history_cnn.history['accuracy']
val_acc = history_cnn.history['val_accuracy']
loss = history_cnn.history['loss']
val_loss = history_cnn.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(14, 5))

# Gráfico de Precisión
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Entrenamiento (Accuracy)')
plt.plot(epochs_range, val_acc, label='Validación (Val Accuracy)')
plt.legend(loc='lower right')
plt.title('Precisión de Entrenamiento y Validación')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.grid(True, linestyle='--', alpha=0.5)

# Gráfico de Pérdida
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Entrenamiento (Loss)')
plt.plot(epochs_range, val_loss, label='Validación (Val Loss)')
plt.legend(loc='upper right')
plt.title('Pérdida de Entrenamiento y Validación')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Arquitectura propuesta 2

### Diseño: conexiones residuales y aumento de datos

La CNN base obtuvo **66,59 % de accuracy en prueba** y mostró oscilaciones en validación durante sus 10 épocas. Esta observación motivó probar una arquitectura con mayor capacidad y conexiones residuales. No se midieron los gradientes, por lo que no podemos atribuir el resultado de la CNN base a gradientes desvanecientes.

La segunda arquitectura incorpora bloques residuales, aumento de datos y reducción de la tasa de aprendizaje cuando la pérdida de validación deja de mejorar. Como se modificaron varios factores a la vez, la comparación mide la mejora del **conjunto de cambios**, no el efecto aislado de cada uno.

In [ ]:
def residual_block(x, filters):
    shortcut = x
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    # Ajuste de dimensiones del shortcut si cambian los canales
    if shortcut.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters, 1, padding="same", use_bias=False)(shortcut)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    x = tf.keras.layers.add([x, shortcut])
    x = tf.keras.layers.ReLU()(x)
    return x

def build_improved_cnn(input_shape=(64, 64, 3), num_classes=10):
    inputs = tf.keras.Input(shape=input_shape)
    x = tf.keras.layers.Rescaling(1./255)(inputs)
    x = data_augmentation(x)

    x = tf.keras.layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = residual_block(x, 32)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = residual_block(x, 64)
    x = tf.keras.layers.MaxPooling2D(2)(x)

    x = residual_block(x, 128)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs, name="CNN_Residual")

In [ ]:
improved_cnn = build_improved_cnn(num_classes=len(class_names))
build_improved_cnn(input_shape=(64, 64, 3), num_classes=10).summary()

In [ ]:
from tensorflow.keras import callbacks

custom_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

# Compilación
improved_cnn.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

EPOCHS = 25

history_improved = improved_cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=custom_callbacks,
    verbose=1
)

In [ ]:
test_loss_imp, test_acc_imp = improved_cnn.evaluate(test_ds)
print(f"\nExactitud en Test (CNN Mejorada con Residuales): {test_acc_imp * 100:.2f}%")
print(f"Pérdida en Test: {test_loss_imp:.4f}")

# Gráfico comparativo de curvas de aprendizaje
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 5))

# Precisión
plt.subplot(1, 2, 1)
plt.plot(history_cnn.history['val_accuracy'], '--', label='Baseline CNN (Val Acc)')
plt.plot(history_improved.history['val_accuracy'], '-', label='CNN Mejorada Residual (Val Acc)', linewidth=2)
plt.title('Comparación de Val Accuracy: Baseline vs Mejorada')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend()
plt.grid(True, alpha=0.3)

# Pérdida
plt.subplot(1, 2, 2)
plt.plot(history_cnn.history['val_loss'], '--', label='Baseline CNN (Val Loss)')
plt.plot(history_improved.history['val_loss'], '-', label='CNN Mejorada Residual (Val Loss)', linewidth=2)
plt.title('Comparación de Val Loss: Baseline vs Mejorada')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Transfer learning
Solucion a partir de una arquitectura pre entrenada

Utilizare el modelo pre entrenado ResNet-50, el cual es una CNN de 50 capas de profundidad creada por Microsoft Research en 2015

Entrenado con mas de un millon de imagenes de la base de datos de ImageNet y puede clasificar imagenes en 1000 categorias diferentes.

### Implementación de transfer learning con ResNet50

Se usa la base preentrenada en ImageNet sin su clasificador original y una cabeza nueva para las 10 clases. Primero se congela la base; después se descongelan sus últimas 25 capas con una tasa de aprendizaje menor. Las imágenes se mantienen en 64 × 64, igual que en la ejecución cuyos resultados se conservan abajo. `preprocess_input` prepara los valores RGB para ResNet50; esta rama no utiliza el reescalado 1/255 de las CNN propias.

In [ ]:
resnet_base = tf.keras.applications.ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(64, 64, 3)
)
resnet_base.trainable = False

# La entrada y la base tienen ambas dimensiones 64 × 64. Esta definición
# corresponde a la arquitectura mostrada en la salida de la ejecución guardada.
model_tl = tf.keras.Sequential([
    tf.keras.layers.Input((64, 64, 3)),
    tf.keras.layers.Lambda(tf.keras.applications.resnet50.preprocess_input),
    resnet_base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model_tl.summary()

In [ ]:
# Compilar el modelo con Transfer Learning (Fase 1: Feature Extraction)
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_FASE1 = 5

# Entrenar por 5 épocas monitoreando sobre val_ds
history_tl = model_tl.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FASE1,
    verbose=1
)

# Evaluar rendimiento intermedio de la fase 1 en Test para medir la mejora posterior
loss_fase1, acc_fase1 = model_tl.evaluate(test_ds, verbose=0)
print(f"\nExactitud en Test tras Fase 1 (Feature Extraction): {acc_fase1 * 100:.2f}%")

In [ ]:
# transfer learning fase 2: Fine-Tuning (Descongelar las capas superiores de ResNet-50)

resnet_base.trainable = True

# Descongelar las últimas 25 capas de la base (no todas son convolucionales)
for layer in resnet_base.layers[:-25]:
    layer.trainable = False

# Recompilar con un Learning Rate mucho más bajo para no destruir los pesos previos
model_tl.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Entrenar hasta 8 épocas adicionales con EarlyStopping
history_tl_fine = model_tl.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1)
    ],
    verbose=1
)

In [ ]:
# 1. Evaluación final en Test tras Fine-Tuning
test_loss_tl, test_acc_tl = model_tl.evaluate(test_ds)
print(f"\n==================================================")
print(f"Exactitud Final en Test (ResNet-50 Fine-Tuned): {test_acc_tl * 100:.2f}%")
print(f"Pérdida Final en Test: {test_loss_tl:.4f}")
print(f"Mejora por Fine-Tuning: +{(test_acc_tl - acc_fase1) * 100:.2f}%")
print(f"==================================================")

# 2. Concatenar métricas de Fase 1 y Fase 2
acc = history_tl.history['accuracy'] + history_tl_fine.history['accuracy']
val_acc = history_tl.history['val_accuracy'] + history_tl_fine.history['val_accuracy']
loss = history_tl.history['loss'] + history_tl_fine.history['loss']
val_loss = history_tl.history['val_loss'] + history_tl_fine.history['val_loss']

plt.figure(figsize=(14, 5))

# Gráfico de Precisión
plt.subplot(1, 2, 1)
plt.plot(acc, label='Entrenamiento (Accuracy)')
plt.plot(val_acc, label='Validación (Val Accuracy)')
plt.axvline(x=EPOCHS_FASE1 - 1, color='red', linestyle='--', label='Inicio Fine-Tuning')
plt.title('ResNet-50: Precisión (Feature Extraction + Fine-Tuning)')
plt.xlabel('Época')
plt.ylabel('Precisión')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)

# Gráfico de Pérdida
plt.subplot(1, 2, 2)
plt.plot(loss, label='Entrenamiento (Loss)')
plt.plot(val_loss, label='Validación (Val Loss)')
plt.axvline(x=EPOCHS_FASE1 - 1, color='red', linestyle='--', label='Inicio Fine-Tuning')
plt.title('ResNet-50: Pérdida (Feature Extraction + Fine-Tuning)')
plt.xlabel('Época')
plt.ylabel('Pérdida')
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Evaluacion y comparativa.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 1. Obtener etiquetas reales y predicciones para los tres modelos en el Test Set
y_true = []
y_pred_cnn = []
y_pred_imp = []
y_pred_tl = []

print("Calculando predicciones en el conjunto de prueba para los 3 modelos...")
for images, labels in test_ds:
    y_true.extend(labels.numpy())

    # 1. Baseline CNN
    preds_cnn = cnn.predict(images, verbose=0)
    y_pred_cnn.extend(np.argmax(preds_cnn, axis=1))

    # 2. CNN Mejorada (Residual)
    preds_imp = improved_cnn.predict(images, verbose=0)
    y_pred_imp.extend(np.argmax(preds_imp, axis=1))

    # 3. ResNet-50 (Transfer Learning)
    preds_tl = model_tl.predict(images, verbose=0)
    y_pred_tl.extend(np.argmax(preds_tl, axis=1))

y_true = np.array(y_true)
y_pred_cnn = np.array(y_pred_cnn)
y_pred_imp = np.array(y_pred_imp)
y_pred_tl = np.array(y_pred_tl)

print("¡Predicciones de los 3 modelos listas!")

In [ ]:
# 1. Generar reportes de clasificación de sklearn
report_cnn = classification_report(y_true, y_pred_cnn, target_names=class_names, output_dict=True)
report_imp = classification_report(y_true, y_pred_imp, target_names=class_names, output_dict=True)
report_tl = classification_report(y_true, y_pred_tl, target_names=class_names, output_dict=True)

# 2. Función para contar parámetros reales de cada modelo
def count_params_str(model):
    trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
    total = sum(tf.size(w).numpy() for w in model.weights)
    return f"{total:,}", f"{trainable:,}"

tot_cnn, train_cnn = count_params_str(cnn)
tot_imp, train_imp = count_params_str(improved_cnn)
tot_tl, train_tl   = count_params_str(model_tl)

# 3. Tabla comparativa global consolidada
comparativa_global = pd.DataFrame({
    'Métrica / Atributo': [
        'Accuracy (Test)',
        'Macro F1-Score',
        'Test Loss',
        'Parámetros Totales',
        'Parámetros Entrenables',
        'Tamaño del Modelo'
    ],
    '1. CNN Baseline': [
        f"{report_cnn['accuracy'] * 100:.2f}%",
        f"{report_cnn['macro avg']['f1-score'] * 100:.2f}%",
        f"{test_loss:.4f}",
        tot_cnn,
        train_cnn,
        'Muy Liviano (~0.5 MB)'
    ],
    '2. CNN Residual': [
        f"{report_imp['accuracy'] * 100:.2f}%",
        f"{report_imp['macro avg']['f1-score'] * 100:.2f}%",
        f"{test_loss_imp:.4f}",
        tot_imp,
        train_imp,
        'Liviano (~1.2 MB)'
    ],
    '3. ResNet-50 (Fine-Tuned)': [
        f"{report_tl['accuracy'] * 100:.2f}%",
        f"{report_tl['macro avg']['f1-score'] * 100:.2f}%",
        f"{test_loss_tl:.4f}",
        tot_tl,
        train_tl,
        'Pesado (~95 MB)'
    ]
})

print("=== COMPARATIVA GLOBAL MULTI-MODELO ===")
display(comparativa_global)

# 4. Desglose de F1-Score por clase
metricas_por_clase = []
for cls in class_names:
    metricas_por_clase.append({
        'Clase': cls,
        'F1 Baseline': report_cnn[cls]['f1-score'],
        'F1 Residual': report_imp[cls]['f1-score'],
        'F1 ResNet-50': report_tl[cls]['f1-score']
    })

df_clases = pd.DataFrame(metricas_por_clase)
print("\n=== COMPARATIVA DETALLADA F1-SCORE POR CLASE ===")
display(df_clases)

### Matrices de confusión y principales errores por clase
Cada matriz está normalizada por clase real. La tabla muestra además los cinco errores más frecuentes de cada modelo en cantidades absolutas y como porcentaje de la clase.


In [ ]:
# Matrices de confusión sobre el mismo conjunto de prueba para los tres modelos.
# Filas = clase real; columnas = clase predicha.
predictions = {
    'CNN base': y_pred_cnn,
    'CNN residual': y_pred_imp,
    'ResNet50': y_pred_tl,
}
fig, axes = plt.subplots(1, 3, figsize=(24, 7), constrained_layout=True)
error_rows = []

for ax, (model_name, predicted) in zip(axes, predictions.items()):
    matrix = confusion_matrix(y_true, predicted, labels=np.arange(num_classes))
    matrix_normalized = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)
    sns.heatmap(matrix_normalized, annot=True, fmt='.2f', cmap='Blues',
                vmin=0, vmax=1, square=True, cbar=False,
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set(title=model_name, xlabel='Predicción', ylabel='Clase real')
    ax.tick_params(axis='x', rotation=60)

    for actual in range(num_classes):
        for predicted_class in range(num_classes):
            if actual != predicted_class and matrix[actual, predicted_class] > 0:
                error_rows.append({
                    'Modelo': model_name,
                    'Clase real': class_names[actual],
                    'Predicha como': class_names[predicted_class],
                    'Errores': int(matrix[actual, predicted_class]),
                    'Porcentaje de la clase': 100 * matrix_normalized[actual, predicted_class],
                })

plt.show()
errors_df = pd.DataFrame(error_rows)
top_errors = (errors_df.sort_values(['Modelo', 'Errores'], ascending=[True, False])
              .groupby('Modelo', sort=False).head(5).reset_index(drop=True))
print('Cinco confusiones más frecuentes de cada modelo:')
display(top_errors)

### 5. Conclusiones y selección de la solución final

**Resultados de la ejecución guardada :**

| Modelo | Accuracy en prueba | F1 macro | Parámetros totales | Parámetros entrenables en la evaluación final |
|:--|--:|--:|--:|--:|
| CNN base (sin data aug) | 66,59 % | 64,95 % | 111.722 | 111.274 |
| CNN residual propia | 90,15 % | 90,12 % | 309.994 | 308.650 |
| ResNet50 ajustada | 91,59 % | 91,47 % | 24.114.826 | 10.517.258 |

La arquitectura residual supera a la base en **23,56 puntos porcentuales** de accuracy. La diferencia reúne cambios de arquitectura, aumento de datos y estrategia de entrenamiento; este experimento no permite atribuirla solamente a los atajos residuales. ResNet50 supera a la residual en **1,44 puntos** de accuracy y **1,35 puntos** de F1 macro.

**Generalización y costo.** Las curvas muestran oscilaciones de validación en la CNN base y la residual. En la segunda fase de ResNet50, la accuracy de entrenamiento llega aproximadamente al 99 %, frente a cerca del 91 % en validación: hay indicios de overfitting. ResNet50 tiene unas **78 veces** más parámetros totales y **34 veces** más parámetros entrenables en el ajuste final que la CNN residual. Esta comparación expresa costo en cantidad de parámetros; no se midieron tiempos de entrenamiento o energía de manera controlada.

**Errores entre clases.** Las matrices de confusión y la tabla de pares de errores de la sección anterior permiten identificar las confusiones dominantes de cada modelo. Se deben interpretar los pares mostrados por esa celda después de ejecutar el notebook completo; no se infieren pares concretos solo a partir del F1 por clase.

**Elección.** Para un entorno con memoria o cómputo limitados elegiríamos la CNN residual: obtiene 90,15 % de accuracy con unos 310.000 parámetros. Si se prioriza la mejor métrica observada y se dispone de más recursos, elegiríamos ResNet50.